<a href="https://colab.research.google.com/github/ymuto0302/PJ2025/blob/main/age_estimation_UTKFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 顔画像からの年齢推定
<font color="red">注意！！ データセットをアップロードする前にランタイムを GPU へ切り替えること！</font>

### UTKFace (aligned & cropped) dataset の置き場所：
https://drive.google.com/drive/folders/1iUv7j7A-ND7pkrTtyLUkYCEBbhV8FveI?usp=sharing

### 設定したクラス：  
0〜4歳，5〜9歳，10代，20代，30代，40代, 50代，60代, 70代, 80歳以上の10クラス

In [1]:
# UTKFace dataset の解凍 (for Google Colab)
!unzip archive.zip -d UTKFace >& /dev/null

In [2]:
# グラフにて日本語を使うために (for Google Colab)
!pip install japanize_matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 109.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for japanize_matplotlib: filename=japanize_matplotlib-1.1.3-py3-none-any.whl size=4120257 sha256=025ce71f49bba3d221f3f24013c11d447b61795b33b002dad9abae1390af887f
  Stored in directory: /root/.cache/pip/wheels/c1/f7/9b/418f19a7b9340fc16e071e89efc379aca68d40238b258df53d
Successfully built japanize_matplotlib


In [14]:
import os
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import japanize_matplotlib

from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T

from tqdm import tqdm


In [4]:
# ==============================
# 0. 設定
# ==============================
class Config:
    data_dir    = "UTKFace/utkface_aligned_cropped/UTKFace"
    image_size  = 224 # 128 # ResNetの想定するサイズに変更
    val_ratio   = 0.2
    seed        = 42

    epochs      = 30
    batch_size  = 64
    lr          = 1e-3
    weight_decay= 1e-4

    num_classes = 10
    save_path   = "best_age_cls.pth"
    cm_path     = "confusion_matrix.png"

# 年齢グループ定義: (ラベル名, 下限, 上限, クラスID)
AGE_GROUPS = [
    ("0-4歳",    0,   4, 0),
    ("5-9歳",    5,   9, 1),
    ("10代",    10,  19, 2),
    ("20代",    20,  29, 3),
    ("30代",    30,  39, 4),
    ("40代",    40,  49, 5),
    ("50代",    50,  59, 6),
    ("60代",    60,  69, 7),
    ("70代",    70,  79, 8),
    ("80歳以上", 80, 999, 9),
]
CLASS_NAMES = [g[0] for g in AGE_GROUPS]

In [5]:
# ==============================
# 1. 年齢 → クラスID の変換
# ==============================

def age_to_class(age: int) -> int | None:
    for _, lo, hi, cid in AGE_GROUPS:
        if lo <= age <= hi:
            return cid
    return None  # 対象外 (100歳超など)


In [6]:
# ==============================
# 2. データセット
# ==============================

class UTKFaceDataset(Dataset):
    def __init__(self, samples: list[tuple[str, int]], transform=None):
        self.samples   = samples  # [(path, class_id), ...]
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def build_samples(data_dir: str) -> list[tuple[str, int]]:
    samples = []
    for fname in os.listdir(data_dir):
        parts = fname.split("_")
        if len(parts) < 4:
            continue
        try:
            age = int(parts[0])
        except ValueError:
            continue
        cid = age_to_class(age)
        if cid is None:
            continue
        samples.append((os.path.join(data_dir, fname), cid))
    return samples


def split_samples(
    samples: list[tuple[str, int]],
    val_ratio: float,
    seed: int,
) -> tuple[list, list]:
    rng = random.Random(seed)
    shuffled = samples[:]
    rng.shuffle(shuffled)
    n_val = int(len(shuffled) * val_ratio)
    return shuffled[n_val:], shuffled[:n_val]


In [7]:
# ==============================
# 3. データ変換
# ==============================

# ImageNet 統計 (pretrained ResNet に合わせる)
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def get_transforms(image_size: int, train: bool) -> T.Compose:
    if train:
        return T.Compose([
            T.Resize((image_size + 16, image_size + 16)),
            T.RandomCrop(image_size),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            T.ToTensor(),
            T.Normalize(_MEAN, _STD),
        ])
    return T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])


In [8]:
# ==============================
# 4. モデル (ResNet-18 + 10クラス分類ヘッド)
# ==============================

def build_model(num_classes: int) -> nn.Module:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


In [15]:
# ==============================
# 5. 学習ループ
# ==============================

def train(model, loader, optimizer, criterion, device) -> float:
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in tqdm(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


In [16]:
# ==============================
# 6. 検証
# ==============================

@torch.no_grad()
def validate(model, loader, criterion, device) -> tuple[float, float]:
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in tqdm(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


In [11]:
# ==============================
# 7. 混同行列の保存
# ==============================

@torch.no_grad()
def save_confusion_matrix(model, loader, device, class_names: list[str], save_path: str):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        preds = model(imgs.to(device)).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    cm = confusion_matrix(all_labels, all_preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    ### font = fm.FontProperties(family="Noto Sans CJK JP")
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right") ###, fontproperties=font)
    ax.set_yticklabels(class_names) ###, fontproperties=font)
    ax.set_xlabel("予測ラベル") ###, fontproperties=font)
    ax.set_ylabel("正解ラベル") ###, fontproperties=font)
    ax.set_title("混同行列 (正規化)") ###, fontproperties=font)

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                    fontsize=8, color=color)

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"混同行列の保存先: {save_path}")

    print("\n--- クラス別精度 ---")
    report = classification_report(all_labels, all_preds, target_names=class_names)
    print(report)


In [17]:
# ==============================
# 8. メイン
# ==============================

def main():
    cfg    = Config()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(cfg.seed)
    print(f"device={device}")

    # データ準備
    all_samples = build_samples(cfg.data_dir)
    train_samples, val_samples = split_samples(all_samples, cfg.val_ratio, cfg.seed)
    print(f"train={len(train_samples)} 枚, val={len(val_samples)} 枚")

    cnt = Counter(s[1] for s in train_samples)
    for label, lo, hi, cid in AGE_GROUPS:
        print(f"  [{cid}] {label:8s}: {cnt[cid]:5d} 枚")

    train_set = UTKFaceDataset(train_samples, get_transforms(cfg.image_size, train=True))
    val_set   = UTKFaceDataset(val_samples,   get_transforms(cfg.image_size, train=False))

    train_loader = DataLoader(train_set, batch_size=cfg.batch_size,
                              shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_set,   batch_size=cfg.batch_size,
                              shuffle=False, num_workers=4)

    # モデル・最適化
    model     = build_model(cfg.num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    # 学習
    best_val_acc = 0.0
    for epoch in range(1, cfg.epochs + 1):
        tr_loss, tr_acc = train(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc = validate(model, val_loader, criterion, device)
        scheduler.step()

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), cfg.save_path)

        print(f"[{epoch:3d}/{cfg.epochs}] "
              f"train loss={tr_loss:.4f} acc={tr_acc:.4f} | "
              f"val loss={va_loss:.4f} acc={va_acc:.4f}")

    print(f"\n最良の val accuracy: {best_val_acc:.4f}")

    # ベストモデルで混同行列を出力
    model.load_state_dict(torch.load(cfg.save_path, map_location=device))
    save_confusion_matrix(model, val_loader, device, CLASS_NAMES, cfg.cm_path)


if __name__ == "__main__":
    main()


device=cuda
train=18964 枚, val=4741 枚
  [0] 0-4歳    :  1701 枚
  [1] 5-9歳    :   711 枚
  [2] 10代     :  1219 枚
  [3] 20代     :  5930 枚
  [4] 30代     :  3627 枚
  [5] 40代     :  1789 枚
  [6] 50代     :  1825 枚
  [7] 60代     :  1050 枚
  [8] 70代     :   567 枚
  [9] 80歳以上   :   545 枚


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  0%|          | 0/297 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
100

[  1/30] train loss=1.3021 acc=0.4872 | val loss=1.1627 acc=0.5322


100%|██████████| 75/75 [00:12<00:00,  6.10it/s]


[  2/30] train loss=1.1388 acc=0.5343 | val loss=1.3088 acc=0.5128


...

[ 28/30] train loss=0.3202 acc=0.8798 | val loss=1.4218 acc=0.5824


100%|██████████| 75/75 [00:11<00:00,  6.35it/s]


[ 29/30] train loss=0.3039 acc=0.8884 | val loss=1.4309 acc=0.5851


100%|██████████| 75/75 [00:11<00:00,  6.34it/s]


[ 30/30] train loss=0.2910 acc=0.8950 | val loss=1.4468 acc=0.5788

最良の val accuracy: 0.5931
混同行列の保存先: confusion_matrix.png

--- クラス別精度 ---
              precision    recall  f1-score   support

        0-4歳       0.96      0.87      0.92       466
        5-9歳       0.56      0.64      0.60       184
         10代       0.65      0.59      0.62       312
         20代       0.66      0.80      0.72      1414
         30代       0.47      0.42      0.44       909
         40代       0.37      0.31      0.34       456
         50代       0.49      0.49      0.49       474
         60代       0.46      0.38      0.42       266
         70代       0.34      0.27      0.30       132
       80歳以上       0.69      0.63      0.66       128

    accuracy                           0.59      4741
   macro avg       0.56      0.54      0.55      4741
weighted avg       0.58      0.59      0.59      4741

